# Скрипт-промптинг для догенерации семплов изображений

In [1]:
import os
import base64
import requests
import re
from pathlib import Path
from openai import OpenAI
import time

API(ROUTER AI),Endpoint, Конфигурация генеративной модели

In [2]:
# --- КОНФИГУРАЦИЯ ---
API_KEY = "sk-nRtwfts-ytWr-KHoa7SlaokKrdjUmg9F"
# Укажи правильный URL эндпоинта. Для OpenRouter это обычно:
API_URL = "https://routerai.ru/api/v1"
MODEL_NAME = "google/gemini-3.1-flash-image-preview" # Название твоей модели в Router AI

In [ ]:
'''
INPUT_DIR = Path("input_images")
OUTPUT_DIR = Path("output_images")

# Создаем папки, если их нет
INPUT_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
'''

Промпты 

In [3]:
# Промпт для чата
PROMPT_TEXT_AK = "Сделай сетку 4 на 4 плотно прилегающих друг другу картинок, где каждая будет размером 128 на 128, так, чтобы данная картинка бралась за основу, а на итоговых органично менялся волосиной покров, цвет кожи или веснушки"
PROMPT_TEXT = "Сделай сетку 4 на 4 плотно прилегающих друг другу картинок, где каждая будет размером 128 на 128, так, чтобы данная картинка бралась за основу, а на итоговых органично менялся волосиной покров, цвет кожи или веснушки, варьируй в меру и аккуратно - без перегибов."
'''
prompt_file = "prompt_files.txt"
prompts  = []
with open(prompt_file) as f:
    for prompt in f:
        prompts.append(prompt)
'''

'\nprompt_file = "prompt_files.txt"\nprompts  = []\nwith open(prompt_file) as f:\n    for prompt in f:\n        prompts.append(prompt)\n'

Работа с изображениями

In [4]:
def encode_image(image_path: Path) -> str:
    """Конвертирует локальную картинку в base64 для отправки по API."""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

Формируем запрос к Router API

In [5]:
def chat_with_banano(image_base64: str, file_name: str) -> str:
    """Отправляет запрос в Router AI."""

    client = OpenAI(
    api_key="sk-nRtwfts-ytWr-KHoa7SlaokKrdjUmg9F",
    base_url="https://routerai.ru/api/v1"
    )

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
             {
                "role": "user",
                "content": [
                    {"type": "text", "text": PROMPT_TEXT},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{image_base64}"
                        }
                    }
                ]
            }
        ]
    )
    print(f"[{file_name}] Отправляю запрос в Router AI...")
    return response


In [6]:
def save_base64_image(data_uri: str, save_path: Path) -> bool:
    """Извлекает Base64 данные из строки Data URI и сохраняет как картинку."""
    try:
        # Проверяем, есть ли префикс Data URI, и отрезаем его
        if "base64," in data_uri:
            # Разделяем строку на "data:image/png;" и "iVBORw0KGgo..."
            _, encoded_data = data_uri.split("base64,", 1)
        else:
            # Если префикса нет, значит строка уже чистый base64
            encoded_data = data_uri

        # Расшифровываем строку в байты
        image_data = base64.b64decode(encoded_data)

        # Открываем файл для записи в бинарном режиме ('wb')
        with open(save_path, "wb") as f:
            f.write(image_data)
            
        return True
    except Exception as e:
        print(f"Ошибка при декодировании/сохранении Base64 картинки: {e}")
        return False

Итоговый пайплайн

In [11]:
def run_pipeline(IMAGE_CLASS:str, LIMIT_FOR_GENERATION:int = 20, DELAY:int = 5, START_NUMB_PIC = 0) -> None:
    """Основной цикл пайплайна."""

    INPUT_DIR = Path(f"{IMAGE_CLASS}_images")
    OUTPUT_DIR = Path(f"{IMAGE_CLASS}_generation_output_images")

    # Создаем папки, если их нет
    INPUT_DIR.mkdir(exist_ok=True)
    OUTPUT_DIR.mkdir(exist_ok=True)
    # Получаем все картинки из папки (поддерживаем jpg, png, jpeg)
    valid_extensions = {".jpg", ".jpeg", ".png"}
    images = [f for f in INPUT_DIR.iterdir() if f.suffix.lower() in valid_extensions]
    
    if not images:
        print("Папка input_images пуста! Положите туда картинки.")
        return

    print(f"Найдено {len(images)} картинок. Запускаем пайплайн...\n")

    for numb,img_path in enumerate(images):
        if START_NUMB_PIC > numb:
            continue


        print(f"--- Обработка: {img_path.name} ---")
        
        # 1. Читаем и кодируем
        img_b64 = encode_image(img_path)
        response = chat_with_banano(img_b64,img_path.name)
        # 2. Отправляем в чат
        response_dict = response.model_dump()
        time.sleep(DELAY)

        try:
            # Идем по структуре: choices -> 0 -> message -> images -> 0 -> image_url -> url
            image_data_uri = response_dict["choices"][0]["message"]["images"][0]["image_url"]["url"]
            
            # Обрати внимание: в ответе data:image/png, лучше сохранять с расширением .png
            output_filename = OUTPUT_DIR / f"class_{IMAGE_CLASS}_gen_by_{img_path.name}_.png"

            # Вызываем нашу новую функцию
            success = save_base64_image(image_data_uri, output_filename)
            
            if success:
                print(f"✅ Успешно сохранено: {output_filename}\n")
            else:
                print(f"❌ Ошибка при сохранении файла.\n")

        except KeyError as e:
            # Эта ошибка выскочит, если в ответе вдруг не окажется нужного ключа
            print(f"❌ Не удалось найти картинку в ответе. Отсутствует ключ: {e}")

        if numb == LIMIT_FOR_GENERATION:
            print(f"Из {IMAGE_CLASS} было последовательно обработано {numb} картинок ." )
            break


RUN !

CLASS AF

In [9]:
run_pipeline("AK")

Найдено 867 картинок. Запускаем пайплайн...

--- Обработка: ISIC_0024468.jpg ---
[ISIC_0024468.jpg] Отправляю запрос в Router AI...
✅ Успешно сохранено: AK_generation_output_images\class_AK_0_.png

--- Обработка: ISIC_0024470.jpg ---
[ISIC_0024470.jpg] Отправляю запрос в Router AI...
✅ Успешно сохранено: AK_generation_output_images\class_AK_1_.png

--- Обработка: ISIC_0024511.jpg ---
[ISIC_0024511.jpg] Отправляю запрос в Router AI...
✅ Успешно сохранено: AK_generation_output_images\class_AK_2_.png

--- Обработка: ISIC_0024646.jpg ---
[ISIC_0024646.jpg] Отправляю запрос в Router AI...
✅ Успешно сохранено: AK_generation_output_images\class_AK_3_.png

--- Обработка: ISIC_0024654.jpg ---
[ISIC_0024654.jpg] Отправляю запрос в Router AI...
❌ Не удалось найти картинку в ответе. Отсутствует ключ: 'images'
--- Обработка: ISIC_0024707.jpg ---
[ISIC_0024707.jpg] Отправляю запрос в Router AI...
✅ Успешно сохранено: AK_generation_output_images\class_AK_5_.png

--- Обработка: ISIC_0024763.jpg ---
[I

In [8]:
run_pipeline("SCC")

Найдено 628 картинок. Запускаем пайплайн...

--- Обработка: ISIC_0024329.jpg ---
[ISIC_0024329.jpg] Отправляю запрос в Router AI...
✅ Успешно сохранено: SCC_generation_output_images\class_SCC_0_.png

--- Обработка: ISIC_0024372.jpg ---
[ISIC_0024372.jpg] Отправляю запрос в Router AI...
✅ Успешно сохранено: SCC_generation_output_images\class_SCC_1_.png

--- Обработка: ISIC_0024418.jpg ---
[ISIC_0024418.jpg] Отправляю запрос в Router AI...
✅ Успешно сохранено: SCC_generation_output_images\class_SCC_2_.png

--- Обработка: ISIC_0024450.jpg ---
[ISIC_0024450.jpg] Отправляю запрос в Router AI...
✅ Успешно сохранено: SCC_generation_output_images\class_SCC_3_.png

--- Обработка: ISIC_0024463.jpg ---
[ISIC_0024463.jpg] Отправляю запрос в Router AI...
✅ Успешно сохранено: SCC_generation_output_images\class_SCC_4_.png

--- Обработка: ISIC_0024517.jpg ---
[ISIC_0024517.jpg] Отправляю запрос в Router AI...
✅ Успешно сохранено: SCC_generation_output_images\class_SCC_5_.png

--- Обработка: ISIC_00245

In [10]:
run_pipeline("VASC")

Найдено 253 картинок. Запускаем пайплайн...

--- Обработка: ISIC_0024370.jpg ---
[ISIC_0024370.jpg] Отправляю запрос в Router AI...
✅ Успешно сохранено: VASC_generation_output_images\class_VASC_gen_by_ISIC_0024370.jpg_.png

--- Обработка: ISIC_0024375.jpg ---
[ISIC_0024375.jpg] Отправляю запрос в Router AI...
✅ Успешно сохранено: VASC_generation_output_images\class_VASC_gen_by_ISIC_0024375.jpg_.png

--- Обработка: ISIC_0024402.jpg ---
[ISIC_0024402.jpg] Отправляю запрос в Router AI...
✅ Успешно сохранено: VASC_generation_output_images\class_VASC_gen_by_ISIC_0024402.jpg_.png

--- Обработка: ISIC_0024475.jpg ---
[ISIC_0024475.jpg] Отправляю запрос в Router AI...
❌ Не удалось найти картинку в ответе. Отсутствует ключ: 'images'
--- Обработка: ISIC_0024662.jpg ---
[ISIC_0024662.jpg] Отправляю запрос в Router AI...
✅ Успешно сохранено: VASC_generation_output_images\class_VASC_gen_by_ISIC_0024662.jpg_.png

--- Обработка: ISIC_0024669.jpg ---
[ISIC_0024669.jpg] Отправляю запрос в Router AI...


In [12]:
run_pipeline("DF")

Найдено 239 картинок. Запускаем пайплайн...

--- Обработка: ISIC_0024318.jpg ---
[ISIC_0024318.jpg] Отправляю запрос в Router AI...
✅ Успешно сохранено: DF_generation_output_images\class_DF_gen_by_ISIC_0024318.jpg_.png

--- Обработка: ISIC_0024330.jpg ---
[ISIC_0024330.jpg] Отправляю запрос в Router AI...
✅ Успешно сохранено: DF_generation_output_images\class_DF_gen_by_ISIC_0024330.jpg_.png

--- Обработка: ISIC_0024386.jpg ---
[ISIC_0024386.jpg] Отправляю запрос в Router AI...
✅ Успешно сохранено: DF_generation_output_images\class_DF_gen_by_ISIC_0024386.jpg_.png

--- Обработка: ISIC_0024396.jpg ---
[ISIC_0024396.jpg] Отправляю запрос в Router AI...
✅ Успешно сохранено: DF_generation_output_images\class_DF_gen_by_ISIC_0024396.jpg_.png

--- Обработка: ISIC_0024553.jpg ---
[ISIC_0024553.jpg] Отправляю запрос в Router AI...
✅ Успешно сохранено: DF_generation_output_images\class_DF_gen_by_ISIC_0024553.jpg_.png

--- Обработка: ISIC_0024845.jpg ---
[ISIC_0024845.jpg] Отправляю запрос в Router